In [ ]:
!pip install xgboost mlflow scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

np.random.seed(42)
highways = ['NH-44', 'NH-48', 'NH-16', 'NH-275', 'NH-65']

# ── Source 1: Original CSV data ──────────────────────────────────────────
csv_data = [
    # NH-44
    {'highway':'NH-44','rainfall_mm':85.2,'temp_c':28,'humidity':85,'wind_kph':20,'congestion_level':0.3,'news_risk_count':2,'month':1,'disruption':1},
    {'highway':'NH-44','rainfall_mm':12.0,'temp_c':31,'humidity':60,'wind_kph':10,'congestion_level':0.1,'news_risk_count':0,'month':1,'disruption':0},
    {'highway':'NH-44','rainfall_mm':90.5,'temp_c':27,'humidity':88,'wind_kph':25,'congestion_level':0.4,'news_risk_count':2,'month':2,'disruption':1},
    {'highway':'NH-44','rainfall_mm':5.0,'temp_c':33,'humidity':45,'wind_kph':8,'congestion_level':0.1,'news_risk_count':0,'month':3,'disruption':0},
    {'highway':'NH-44','rainfall_mm':110.3,'temp_c':26,'humidity':92,'wind_kph':30,'congestion_level':0.6,'news_risk_count':3,'month':5,'disruption':1},
    {'highway':'NH-44','rainfall_mm':95.0,'temp_c':25,'humidity':90,'wind_kph':28,'congestion_level':0.5,'news_risk_count':2,'month':6,'disruption':1},
    {'highway':'NH-44','rainfall_mm':88.0,'temp_c':26,'humidity':87,'wind_kph':22,'congestion_level':0.4,'news_risk_count':2,'month':6,'disruption':1},
    {'highway':'NH-44','rainfall_mm':120.5,'temp_c':24,'humidity':95,'wind_kph':35,'congestion_level':0.7,'news_risk_count':3,'month':7,'disruption':1},
    {'highway':'NH-44','rainfall_mm':75.0,'temp_c':27,'humidity':83,'wind_kph':18,'congestion_level':0.3,'news_risk_count':1,'month':7,'disruption':1},
    {'highway':'NH-44','rainfall_mm':0.0,'temp_c':38,'humidity':30,'wind_kph':5,'congestion_level':0.1,'news_risk_count':0,'month':4,'disruption':0},
    # NH-48
    {'highway':'NH-48','rainfall_mm':10.0,'temp_c':29,'humidity':55,'wind_kph':12,'congestion_level':0.2,'news_risk_count':0,'month':1,'disruption':0},
    {'highway':'NH-48','rainfall_mm':80.0,'temp_c':27,'humidity':86,'wind_kph':22,'congestion_level':0.4,'news_risk_count':1,'month':6,'disruption':1},
    {'highway':'NH-48','rainfall_mm':70.0,'temp_c':28,'humidity':84,'wind_kph':20,'congestion_level':0.3,'news_risk_count':1,'month':6,'disruption':1},
    {'highway':'NH-48','rainfall_mm':90.0,'temp_c':26,'humidity':89,'wind_kph':26,'congestion_level':0.5,'news_risk_count':2,'month':8,'disruption':1},
    {'highway':'NH-48','rainfall_mm':5.0,'temp_c':35,'humidity':40,'wind_kph':8,'congestion_level':0.1,'news_risk_count':0,'month':3,'disruption':0},
    # NH-16
    {'highway':'NH-16','rainfall_mm':80.0,'temp_c':27,'humidity':86,'wind_kph':22,'congestion_level':0.4,'news_risk_count':2,'month':6,'disruption':1},
    {'highway':'NH-16','rainfall_mm':95.0,'temp_c':26,'humidity':90,'wind_kph':28,'congestion_level':0.5,'news_risk_count':2,'month':6,'disruption':1},
    {'highway':'NH-16','rainfall_mm':100.0,'temp_c':25,'humidity':92,'wind_kph':30,'congestion_level':0.6,'news_risk_count':3,'month':7,'disruption':1},
    {'highway':'NH-16','rainfall_mm':140.0,'temp_c':24,'humidity':96,'wind_kph':40,'congestion_level':0.8,'news_risk_count':3,'month':11,'disruption':1},
    {'highway':'NH-16','rainfall_mm':10.0,'temp_c':29,'humidity':55,'wind_kph':10,'congestion_level':0.1,'news_risk_count':0,'month':1,'disruption':0},
    # NH-275
    {'highway':'NH-275','rainfall_mm':85.0,'temp_c':24,'humidity':88,'wind_kph':20,'congestion_level':0.3,'news_risk_count':1,'month':6,'disruption':1},
    {'highway':'NH-275','rainfall_mm':90.0,'temp_c':23,'humidity':90,'wind_kph':22,'congestion_level':0.4,'news_risk_count':2,'month':6,'disruption':1},
    {'highway':'NH-275','rainfall_mm':100.0,'temp_c':22,'humidity':93,'wind_kph':28,'congestion_level':0.5,'news_risk_count':2,'month':7,'disruption':1},
    {'highway':'NH-275','rainfall_mm':110.0,'temp_c':23,'humidity':94,'wind_kph':30,'congestion_level':0.6,'news_risk_count':3,'month':7,'disruption':1},
    {'highway':'NH-275','rainfall_mm':5.0,'temp_c':26,'humidity':50,'wind_kph':8,'congestion_level':0.1,'news_risk_count':0,'month':2,'disruption':0},
    # NH-65
    {'highway':'NH-65','rainfall_mm':75.0,'temp_c':27,'humidity':84,'wind_kph':18,'congestion_level':0.3,'news_risk_count':1,'month':6,'disruption':1},
    {'highway':'NH-65','rainfall_mm':95.0,'temp_c':26,'humidity':90,'wind_kph':26,'congestion_level':0.5,'news_risk_count':2,'month':7,'disruption':1},
    {'highway':'NH-65','rainfall_mm':8.0,'temp_c':30,'humidity':50,'wind_kph':10,'congestion_level':0.1,'news_risk_count':0,'month':3,'disruption':0},
    {'highway':'NH-65','rainfall_mm':60.0,'temp_c':28,'humidity':78,'wind_kph':15,'congestion_level':0.2,'news_risk_count':1,'month':5,'disruption':1},
    {'highway':'NH-65','rainfall_mm':0.0,'temp_c':38,'humidity':28,'wind_kph':5,'congestion_level':0.1,'news_risk_count':0,'month':4,'disruption':0},
]
df_csv = pd.DataFrame(csv_data)
df_csv['highway_id'] = df_csv['highway'].apply(lambda x: highways.index(x))
df_csv = df_csv.drop('highway', axis=1)

# ── Source 2: Synthetic with seasonal patterns ────────────────────────────
records = []
for _ in range(600):
    highway = np.random.choice(highways)
    month = np.random.randint(1, 13)

    # Seasonal rainfall bias
    if month in [6, 7, 8, 9]:
        rainfall = np.random.choice([
            np.random.uniform(0, 20),
            np.random.uniform(20, 60),
            np.random.uniform(60, 120),
            np.random.uniform(120, 200)
        ], p=[0.1, 0.2, 0.4, 0.3])
        humidity = np.random.uniform(70, 98)
    elif month in [10, 11]:
        rainfall = np.random.choice([
            np.random.uniform(0, 10),
            np.random.uniform(10, 40),
            np.random.uniform(40, 100),
            np.random.uniform(100, 180)
        ], p=[0.2, 0.3, 0.3, 0.2])
        humidity = np.random.uniform(60, 92)
    else:
        rainfall = np.random.choice([
            np.random.uniform(0, 5),
            np.random.uniform(5, 20),
            np.random.uniform(20, 50),
        ], p=[0.6, 0.3, 0.1])
        humidity = np.random.uniform(30, 65)

    congestion = np.random.uniform(0, 1)
    news_risk = np.random.randint(0, 4)
    temp = np.random.uniform(18, 42)
    wind = np.random.uniform(0, 60)

    # Highway specific seasonal risk
    highway_season_bonus = 0
    if highway == 'NH-44' and month in [6, 7, 8, 9]:
        highway_season_bonus = 0.3
    elif highway == 'NH-16' and month in [10, 11]:
        highway_season_bonus = 0.4
    elif highway == 'NH-275' and month in [6, 7, 8, 9]:
        highway_season_bonus = 0.2

    disruption = 0
    if rainfall > 80:
        disruption = 1
    elif rainfall > 40 and humidity > 75:
        disruption = 1
    elif news_risk >= 2:
        disruption = 1
    elif congestion > 0.7:
        disruption = 1
    elif highway_season_bonus > 0 and rainfall > 20:
        disruption = 1

    records.append({
        'highway_id': highways.index(highway),
        'rainfall_mm': round(rainfall, 2),
        'temp_c': round(temp, 2),
        'humidity': round(humidity, 2),
        'wind_kph': round(wind, 2),
        'congestion_level': round(congestion, 2),
        'news_risk_count': news_risk,
        'month': month,
        'highway_season_bonus': round(highway_season_bonus, 2),
        'disruption': disruption
    })

df_synthetic = pd.DataFrame(records)

# Add highway_season_bonus to csv data too
df_csv['highway_season_bonus'] = 0.0

# ── Combine both sources ──────────────────────────────────────────────────
df = pd.concat([df_csv, df_synthetic], ignore_index=True)
print(f"Total records: {len(df)}")
print(df['disruption'].value_counts())
print(df.head())

In [ ]:
X = df.drop('disruption', axis=1)
y = df['disruption']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

mlflow.set_experiment("freight-risk-prediction-v2")

with mlflow.start_run():
    model = xgb.XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("data_sources", "csv+synthetic_seasonal")
    mlflow.log_param("total_records", len(df))
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    print(f"Total training records: {len(df)}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(classification_report(y_test, y_pred))

    # Save feature names for Spark
    feature_names = list(X.columns)
    print(f"Features: {feature_names}")

with open('freight_risk_model_v2.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model v2 saved.")